In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%cudf.pandas.profile
### cell 11 ###

### cell 11 (optimized):
# 1) read all essays into a temporary cudf‐backed DataFrame
ids   = [t.split("/")[-1].replace(".txt", "") for t in train_txt]
texts = [open(t, "r").read()             for t in train_txt]
tmp   = pd.DataFrame({"id": ids, "essay_text": texts})

# 2) compute length and word‐count on the GPU using vectorized string ops
#    then cast to int64 so they match the original DataFrame's dtypes
tmp["essay_len"]   = tmp["essay_text"].str.strip().str.len().astype("int64")
tmp["essay_words"] = tmp["essay_text"].str.split().list.len().astype("int64")

# 3) join back onto the main DataFrame
train = train.merge(
    tmp[["id", "essay_len", "essay_words"]],
    on="id",
    how="left"
)
